In [1]:
from pbase.config.ray import init_ray
import shadow_price_prediction

# Initialize Ray for parallel processing
extra_modules = [shadow_price_prediction]
init_ray(address='ray://10.8.0.36:10001', extra_modules=extra_modules)

import pandas as pd
import numpy as np
import random
from pathlib import Path
from pbase.utils.ray import parallel_equal_pool

# Import tuning utilities for resumable search
from shadow_price_prediction.tuning_utils import (
    load_previous_params,
    sample_params,
    run_single_experiment
)

pd.set_option('display.max_columns', 99)

2025-11-22 23:34:28,011	INFO client_builder.py:242 -- Passing the following kwargs to ray.init() on the server: log_to_driver
2025-11-22 23:34:28,357	INFO packaging.py:588 -- Creating a file package for local module '/home/xyz/workspace/research-spice-shadow-price-pred/src/shadow_price_prediction'.
2025-11-22 23:34:28,366	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_00c06ac0702a0e27.zip' (0.53MiB) to Ray cluster...
2025-11-22 23:34:28,372	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_00c06ac0702a0e27.zip'.


In [2]:
N_ITER = 10000  # Total number of experiments to run (including previous ones)
BASE_OUTPUT_DIR = Path('/opt/temp/haoyan/param_search_new')

## 5. Analyze Best Results

In [3]:
metrics_dir = BASE_OUTPUT_DIR / 'metrics'

if metrics_dir.exists():
    all_metrics = []
    for file in metrics_dir.glob('iter_*.parquet'):
        try:
            df = pd.read_parquet(file)
            all_metrics.append(df)
        except Exception as e:
            print(f"Warning: Could not load {file.name}: {e}")
    
    if all_metrics:
        combined_metrics = pd.concat(all_metrics, ignore_index=True)
        
        print(f"Total runs analyzed: {len(combined_metrics)}")
        
        # Best by F1 Score
        if 'F1' in combined_metrics.columns:
            best_idx = combined_metrics['F1'].idxmax()
            best_result = combined_metrics.iloc[best_idx]
            
            print("\n" + "=" * 60)
            print("Best Run (by F1 Score)")
            print("=" * 60)
            print(f"Iteration: {best_result.get('iteration', 'N/A')}")
            print(f"F1 Score: {best_result['F1']:.4f}")
            if 'MAE' in best_result:
                print(f"MAE: {best_result['MAE']:.4f}")
            if 'Precision' in best_result:
                print(f"Precision: {best_result['Precision']:.4f}")
            if 'Recall' in best_result:
                print(f"Recall: {best_result['Recall']:.4f}")
            
            # Print parameters
            print("\nParameters:")
            param_cols = [col for col in combined_metrics.columns 
                         if col not in ['F1', 'Precision', 'Recall', 'MAE', 'RMSE', 'R2',
                                       'iteration', 'timestamp', 'run_id', 'metrics_file',
                                       'per_outage_file', 'agg_file']]
            for col in param_cols:
                print(f"  {col}: {best_result[col]}")
        
        # Best by MAE (lower is better)
        if 'MAE' in combined_metrics.columns:
            best_idx = combined_metrics['MAE'].idxmin()
            best_result = combined_metrics.iloc[best_idx]
            
            print("\n" + "=" * 60)
            print("Best Run (by MAE)")
            print("=" * 60)
            print(f"Iteration: {best_result.get('iteration', 'N/A')}")
            print(f"MAE: {best_result['MAE']:.4f}")
            if 'F1' in best_result:
                print(f"F1 Score: {best_result['F1']:.4f}")
            if 'RMSE' in best_result:
                print(f"RMSE: {best_result['RMSE']:.4f}")
            
            # Print parameters
            print("\nParameters:")
            for col in param_cols:
                print(f"  {col}: {best_result[col]}")

Total runs analyzed: 588


## 6. View Results Summary

In [ ]:
x = pd.read_parquet(combined_metrics.sort_values('f1_score', ascending=False).loc[3]['agg_file'])
cols = ['90_diff', '105_diff', '95_diff', '100_diff', 'curvature_100', '60_diff', 'prob_overload', '90_diff', '85_diff', '100_diff', '70_diff', '95_diff', 'prob_exceed_100', '80_diff', 'prob_exceed_90', '105', 'log_density_100', '110', 'risk_ratio', 'actual_shadow_price', 'predicted_shadow_price', 'binding_probability', 'predicted_binding_count']
x = x[x['predicted_binding_count'] >= 1].sort_values(['predicted_shadow_price', 'binding_probability', 'predicted_binding_count'], ascending=[False, False, False]).drop_duplicates(subset=['auction_month', 'branch_name'])
print(x.groupby('auction_month').size())
x[cols].corr(method='pearson')['actual_shadow_price'].sort_values(ascending=False)

auction_month
2025-07-01    490
2025-08-01    382
2025-09-01    626
2025-10-01    632
dtype: int64


actual_shadow_price        1.000000
predicted_shadow_price     0.300131
predicted_binding_count    0.181142
binding_probability        0.159086
log_density_100            0.107246
105_diff                   0.098622
105                        0.098395
110                        0.097264
100_diff                   0.097205
100_diff                   0.097205
95_diff                    0.095053
95_diff                    0.095053
90_diff                    0.090839
90_diff                    0.090839
prob_exceed_90             0.088875
85_diff                    0.084683
80_diff                    0.081388
70_diff                    0.079735
prob_overload              0.075787
prob_exceed_100            0.075787
60_diff                    0.067397
curvature_100              0.009775
risk_ratio                -0.016389
Name: actual_shadow_price, dtype: float64

In [ ]:
a = x[x['auction_month']=='2025-10']
a[a['branch_name'].str.contains('aust', case=False)]

,,branch_name,log_density_100,80_diff,prob_exceed_100,curvature_100,100_diff,85_diff,60_diff,100,105,70_diff,90,prob_exceed_90,110,prob_overload,95_diff,risk_ratio,90_diff,105_diff,95,actual_shadow_price,predicted_shadow_price,binding_probability,binding_probability_scaled,threshold,predicted_binding_count,actual_binding,auction_month,market_month,predicted_binding,error,abs_error,model_used
constraint_id,flow_direction,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
316316,1,AUST_TAYS_1545 A,3.769987,2.364259,4.119607,0.135582,3.476634,2.682441,1.146306,3.200974,3.671965,1.787645,2.78819,4.073361,3.827148,4.119607,3.161357,-0.002453,2.913186,3.871775,2.952014,2950.54,7662.726562,0.738490,0.0,0.5,9,1,2025-10-01,2025-10-01,True,4712.186562,4712.186562,Default
316680,1,MAUSTHLT69_1 1,-0.247172,-0.241334,-0.180466,-0.028706,-0.217223,-0.241286,-0.175316,-0.215049,-0.209627,-0.238180,-0.23788,-0.217988,-0.173675,-0.180466,-0.227935,-0.002475,-0.236435,-0.199252,-0.225976,0.00,1429.236694,0.267182,0.0,0.5,2,0,2025-10-01,2025-10-01,True,1429.236694,1429.236694,Default


In [ ]:
if 'combined_metrics' in locals():
    # Display summary statistics
    print("Summary Statistics:")
    print("=" * 60)
    
    metric_cols = ['f1_score', 'precision', 'recall', 'mae', 'rmse']
    available_metrics = [col for col in metric_cols if col in combined_metrics.columns]
    
    summary = combined_metrics[available_metrics].describe()
    print(summary)
    
    # Optionally display the top 5 runs by F1
    if 'F1' in combined_metrics.columns:
        print("\n" + "=" * 60)
        print("Top 5 Runs by F1 Score")
        print("=" * 60)
        
        display_cols = ['iteration'] + available_metrics
        top5 = combined_metrics.nlargest(5, 'F1')[display_cols]
        print(top5.to_string(index=False))

Summary Statistics:
         f1_score   precision      recall         mae         rmse
count  588.000000  588.000000  588.000000  588.000000   588.000000
mean     0.340072    0.329733    0.351464  327.264258  1847.983187
std      0.013587    0.007486    0.020915    6.099367    70.495382
min      0.267990    0.295821    0.244945  313.298755  1745.457301
25%      0.337177    0.325526    0.348291  322.538323  1807.515468
50%      0.343159    0.330924    0.356473  327.209863  1834.725888
75%      0.348021    0.335002    0.363613  331.394778  1865.876581
max      0.359390    0.344974    0.378987  346.003628  2196.036819


In [ ]:
# import shutil

# shutil.rmtree(BASE_OUTPUT_DIR)

In [4]:
import pandas as pd
from shadow_price_prediction.config import PredictionConfig
from shadow_price_prediction.tuning_utils import update_config_with_params
from shadow_price_prediction.pipeline import ShadowPricePipeline

def create_pipeline_from_row(row):
    """
    Create a ShadowPricePipeline using the parameters from a single row of results.
    
    Args:
        row: DataFrame row with parameters
    
    Returns:
        ShadowPricePipeline configured with the parameters
    """
    params_dict = row.to_dict()
        
    print("Initializing pipeline with selected parameters:")
    for k, v in params_dict.items():
        if k.startswith(('clf_', 'reg_', 'threshold_', 'label_')):
             print(f"  {k}: {v}")

    # Create Config
    config = PredictionConfig()
    
    # Update Config with Params
    config = update_config_with_params(config, params_dict)
    
    # Initialize Pipeline
    pipeline = ShadowPricePipeline(config)
    
    return pipeline

In [33]:
params = combined_metrics.sort_values('f1_score', ascending=False).iloc[0]
params['clf_xgb_n_estimators'] = 1000
params['reg_xgb_n_estimators'] = 1000
params['reg_xgb_max_depth'] = 7
params['clf_xgb_max_depth'] = 7
# params['reg_xgb_max_depth'] = 2
# params['clf_xgb_max_depth'] = 2
params['label_threshold'] = 0
params['threshold_beta'] = 0.5
params['short_term_clf_xgb_w'] = 0.9
params['short_term_reg_xgb_w'] = 0.9
params['medium_term_clf_xgb_w'] = 0.7
params['medium_term_reg_xgb_w'] = 0.7
params['long_term_clf_xgb_w'] = 0.9
params['long_term_reg_xgb_w'] = 0.9


"""
clf_xgb_n_estimators: 500
clf_xgb_max_depth: 5
clf_xgb_learning_rate: 0.2
clf_xgb_gamma: 0.1
clf_xgb_min_child_weight: 10
clf_xgb_reg_alpha: 0.0
clf_xgb_reg_lambda: 100
reg_xgb_n_estimators: 500
reg_xgb_max_depth: 5
reg_xgb_learning_rate: 0.1
reg_xgb_gamma: 0.2
reg_xgb_min_child_weight: 10
reg_xgb_reg_alpha: 1.0
reg_xgb_reg_lambda: 1
threshold_beta: 1.0
clf_xgb_thres: 0.5
reg_xgb_thres: 0.5
"""

'\nclf_xgb_n_estimators: 500\nclf_xgb_max_depth: 5\nclf_xgb_learning_rate: 0.2\nclf_xgb_gamma: 0.1\nclf_xgb_min_child_weight: 10\nclf_xgb_reg_alpha: 0.0\nclf_xgb_reg_lambda: 100\nreg_xgb_n_estimators: 500\nreg_xgb_max_depth: 5\nreg_xgb_learning_rate: 0.1\nreg_xgb_gamma: 0.2\nreg_xgb_min_child_weight: 10\nreg_xgb_reg_alpha: 1.0\nreg_xgb_reg_lambda: 1\nthreshold_beta: 1.0\nclf_xgb_thres: 0.5\nreg_xgb_thres: 0.5\n'

In [34]:
pipeline = create_pipeline_from_row(params)

Initializing pipeline with selected parameters:
  clf_xgb_n_estimators: 1000
  clf_xgb_max_depth: 7
  clf_xgb_learning_rate: 0.2
  clf_xgb_gamma: 0.2
  clf_xgb_min_child_weight: 10
  clf_xgb_reg_alpha: 0.1
  clf_xgb_reg_lambda: 100
  reg_xgb_n_estimators: 1000
  reg_xgb_max_depth: 7
  reg_xgb_learning_rate: 0.1
  reg_xgb_gamma: 0.0
  reg_xgb_min_child_weight: 10
  reg_xgb_reg_alpha: 0.0
  reg_xgb_reg_lambda: 100
  threshold_beta: 0.5
  clf_xgb_thres: 0.5
  reg_xgb_thres: 0.5
  label_threshold: 0


In [35]:
test_periods = [
    (pd.Timestamp('2025-06-01'), pd.Timestamp('2025-06-01')),
    (pd.Timestamp('2025-07-01'), pd.Timestamp('2025-07-01')),
    (pd.Timestamp('2025-08-01'), pd.Timestamp('2025-08-01')),
    (pd.Timestamp('2025-09-01'), pd.Timestamp('2025-09-01')),
    (pd.Timestamp('2025-10-01'), pd.Timestamp('2025-10-01')),
    (pd.Timestamp('2025-11-01'), pd.Timestamp('2025-11-01')),
]

results_per_outage, final_results, metrics = pipeline.run(
    test_periods=test_periods,
    verbose=True,
    use_parallel=True,
    n_jobs=6,
)

SHADOW PRICE PREDICTION PIPELINE - PARALLEL PER-AUCTION-MONTH MODE
Total Test Periods: 6
Unique Auction Months: 6

  Auction: 2025-06 → 1 market month(s)
    - Market: 2025-06
  Auction: 2025-07 → 1 market month(s)
    - Market: 2025-07
  Auction: 2025-08 → 1 market month(s)
    - Market: 2025-08
  Auction: 2025-09 → 1 market month(s)
    - Market: 2025-09
  Auction: 2025-10 → 1 market month(s)
    - Market: 2025-10
  Auction: 2025-11 → 1 market month(s)
    - Market: 2025-11
Class Type: onpeak
Period Type: f0

🚀 Using Ray parallel processing for 6 auction months
   Workers: 6

[PARALLEL PROCESSING: Training and Predicting for All Auction Months]


(PoolActor pid=732223, ip=10.42.8.15) 25-11-23 00:07:00 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=732223, ip=10.42.8.15) 25-11-23 00:07:00 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=732223, ip=10.42.8.15) 25-11-23 00:07:00 EST | INFO    | pbase.config.base    | load_global_config  | #435 | Loading config from local cached json file...
(PoolActor pid=732223, ip=10.42.8.15) 25-11-23 00:07:00 EST | INFO    | pbase.config.base    | __init__            | #44  | Init global config...
(PoolActor pid=730680, ip=10.42.9.131) 25-11-23 00:07:00 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=730680, ip=10.42.9.131) 25-11-23 00:07:00 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=730680, i

(PoolActor pid=732223, ip=10.42.8.15) 
(PoolActor pid=732223, ip=10.42.8.15) ================================================================================
(PoolActor pid=732223, ip=10.42.8.15) [Processing Auction Month: 2025-08]
(PoolActor pid=732223, ip=10.42.8.15)   Market Months: 1
(PoolActor pid=732223, ip=10.42.8.15)     - 2025-08
(PoolActor pid=732223, ip=10.42.8.15) ================================================================================
(PoolActor pid=732223, ip=10.42.8.15) 
(PoolActor pid=732223, ip=10.42.8.15) [STEP 1: Training Period Calculation]
(PoolActor pid=732223, ip=10.42.8.15)   Training Range: 2024-07 to 2025-07
(PoolActor pid=732223, ip=10.42.8.15) 
(PoolActor pid=732223, ip=10.42.8.15) [STEP 2: Loading Training Data]
(PoolActor pid=732223, ip=10.42.8.15)   Required period types: ['f0']
(PoolActor pid=732223, ip=10.42.8.15) 
(PoolActor pid=732223, ip=10.42.8.15) [Loading Training Data - Multi-Period Strategy]
(PoolActor pid=732223, ip=10.42.8.15) --------

  0%|          | 0/6 [00:00<?, ?it/s]


[COMBINING RESULTS ACROSS ALL PERIODS]

✓ Results Combined
  Total samples (per-outage): 808,108
  Total unique constraints (monthly): 88,221
  Date range: 2025-06-01 to 2025-11-28

[ANALYZING COMBINED RESULTS]

[MONTHLY AGGREGATED RESULTS]

Test Period: Monthly Level
  Total samples: 88,221

[Shadow Price Prediction Performance]
  MAE:  $844.92
  RMSE: $78057.84
  MAPE: 3861.07%

[Classification Performance]
  Precision: 0.291
  Recall:    0.581
  F1-Score:  0.388
  Accuracy:  0.858

[Binding Classification Details]
  Correctly classified: 75,667 / 88,221 (85.77%)

  True Positives (correctly identified binding): 3,979
  False Positives (incorrectly predicted as binding): 9,682
  True Negatives (correctly identified non-binding): 71,688
  False Negatives (missed binding constraints): 2,872

[Binding Rate Analysis]
  Actual binding rate: 7.77% (6,851 samples)
  Predicted binding rate: 15.48% (13,661 samples)

[PER-OUTAGE SUMMARY]

Total outage dates: 63

Average metrics across outage 

In [36]:
print(pipeline.config.features.step1_features) # For classifiers

['risk_ratio', 'curvature_100', 'prob_exceed_110', 'prob_exceed_105', 'prob_exceed_100', 'prob_exceed_95', 'prob_exceed_90', 'season_hist_da_1', 'season_hist_da_2', 'season_hist_da_3', 'recent_hist_da']


In [37]:
xgb_model = pipeline.trained_models[pd.Timestamp('2025-06-01')].clf_default_ensemble_f0[0][0]

# 2. Get feature names from config
# For classifiers, we use step1_features
feature_names = pipeline.config.features.step1_features

# 3. Create a DataFrame for readability
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df)

             feature  importance
6     prob_exceed_90    0.371392
10    recent_hist_da    0.101025
5     prob_exceed_95    0.068334
7   season_hist_da_1    0.062957
0         risk_ratio    0.062101
4    prob_exceed_100    0.061438
3    prob_exceed_105    0.061030
2    prob_exceed_110    0.059929
1      curvature_100    0.054334
9   season_hist_da_3    0.050763
8   season_hist_da_2    0.046697


In [16]:
# final_results.to_parquet('/opt/temp/haoyan/final_results_test.parquet')
# results_per_outage.to_parquet('/opt/temp/haoyan/results_per_outage_test.parquet')

In [38]:
metrics

{'monthly': {'mae': 844.9164626065061,
  'rmse': np.float64(78057.84285735543),
  'mape': np.float64(3861.074349830882),
  'precision': 0.29126711075323913,
  'recall': 0.5807911253831557,
  'f1_score': 0.3879680187207488,
  'accuracy': np.float64(0.857698280454767),
  'true_positives': 3979,
  'false_positives': 9682,
  'true_negatives': 71688,
  'false_negatives': 2872,
  'total_samples': 88221,
  'actual_binding_count': 6851,
  'predicted_binding_count': 13661,
  'actual_binding_rate': 0.07765724714070346,
  'predicted_binding_rate': 0.15484975232654358},
 'per_outage': {'2025-06-01': {'mae': 66.19675418322161,
   'rmse': np.float64(369.7572313874855),
   'mape': np.float64(163.1254644197319),
   'precision': 0.18361581920903955,
   'recall': 0.45990566037735847,
   'f1_score': 0.26244952893674295,
   'accuracy': np.float64(0.9138500235811979),
   'true_positives': 195,
   'false_positives': 867,
   'true_negatives': 11431,
   'false_negatives': 229,
   'total_samples': 12722,
   'a

In [39]:
sig_list = []
for date in pd.date_range('2025-06', '2025-11', freq='MS'):
    # if date.month in [5, 6]:
    #     continue
    date_str = date.strftime('%Y-%m')
    f1_sig = pd.read_parquet(f'/opt/data/xyz-dataset/signal_data/miso/constraints/TEST.TEST.Signal.MISO.SPICE_F0P_V6.2B.R1/{date_str}/f0/onpeak/')
    f1_sig['auction_month'] = date
    sig_list.append(f1_sig)

sig_df = pd.concat(sig_list)
sig_df.sort_values('tier', ascending=True).drop_duplicates(subset=['auction_month', 'branch_name'], keep='first')

fin_df = final_results.sort_values(['binding_probability_scaled'], ascending=False).drop_duplicates(subset=['auction_month', 'branch_name'], keep='first')

In [40]:
df = fin_df.reset_index().merge(sig_df, on=['branch_name', 'auction_month'], how='outer')
df = df[~df['constraint_id_x'].isin(['SO_MW_Transfer'])]

In [41]:
print(df[(df['actual_shadow_price'] > 1000) & (df['tier'].isna()) & (df['predicted_binding_count'] >= 2)].sort_values('actual_shadow_price').shape)
print(df[(df['actual_shadow_price'] > 1000) & (df['tier'].notna()) & (df['predicted_binding_count'] == 0)].sort_values('actual_shadow_price').shape)
print(df[(df['actual_shadow_price'] > 1000) & (df['tier'].notna()) & (df['predicted_binding_count'] >= 2)].sort_values('actual_shadow_price').shape)
print(df[(df['actual_shadow_price'] > 1000) & (df['predicted_binding_count'] >= 2)].sort_values('actual_shadow_price').shape)
print(df[(df['actual_shadow_price'] > 1000) & (df['tier'].notna())].sort_values('actual_shadow_price').shape)

(61, 46)
(56, 46)
(283, 46)
(344, 46)
(376, 46)


In [42]:
df['sig_pred'] = (df['actual_shadow_price'] > 1000) & (df['tier'] <= 4)
df['sig_has_pred'] = df['tier'] <= 4
ddf = df.groupby('auction_month')[['sig_pred', 'sig_has_pred']].sum()
ddf['sig_pred_rate'] = ddf['sig_pred'] / ddf['sig_has_pred']
ddf.describe()

,sig_pred,sig_has_pred,sig_pred_rate
count,6.000000,6.000000,6.000000
mean,62.666667,585.166667,0.107469
std,17.362795,104.935059,0.021654
min,49.000000,488.000000,0.078404
25%,54.000000,496.500000,0.099230
50%,58.000000,559.000000,0.102254
75%,60.500000,666.500000,0.116966
max,97.000000,727.000000,0.141813


In [43]:
df['sig_pred'] = (df['actual_shadow_price'] > 1000) & (df['predicted_binding_count'] >= 2)
df['sig_has_pred'] = df['predicted_binding_count'] >= 2
df.groupby('auction_month')[['sig_pred', 'sig_has_pred']].sum()
ddf = df.groupby('auction_month')[['sig_pred', 'sig_has_pred']].sum()
ddf['sig_pred_rate'] = ddf['sig_pred'] / ddf['sig_has_pred']
ddf.describe()

,sig_pred,sig_has_pred,sig_pred_rate
count,6.000000,6.000000,6.000000
mean,57.333333,569.333333,0.100603
std,14.348054,85.787334,0.016352
min,47.000000,451.000000,0.076923
25%,51.000000,531.250000,0.095117
50%,52.000000,550.000000,0.101013
75%,55.250000,635.500000,0.103476
max,86.000000,676.000000,0.127219


In [ ]:
metrics

{'monthly': {'mae': 473.9555450442503,
  'rmse': np.float64(3366.3656046556816),
  'mape': np.float64(29498.3000746118),
  'precision': 0.2899688554373216,
  'recall': 0.27880716201884087,
  'f1_score': 0.28427848986991505,
  'accuracy': np.float64(0.8877846150777424),
  'true_positives': 4469,
  'false_positives': 10943,
  'true_negatives': 173562,
  'false_negatives': 11560,
  'total_samples': 200534,
  'actual_binding_count': 16029,
  'predicted_binding_count': 15412,
  'actual_binding_rate': 0.07993158267425973,
  'predicted_binding_rate': 0.07685479769016726},
 'per_outage': {'2024-08-01': {'mae': 43.83626895834911,
   'rmse': np.float64(329.51001796580954),
   'mape': np.float64(159.53908680118045),
   'precision': 0.14717741935483872,
   'recall': 0.26838235294117646,
   'f1_score': 0.19010416666666666,
   'accuracy': np.float64(0.9510621557828481),
   'true_positives': 73,
   'false_positives': 423,
   'true_negatives': 12015,
   'false_negatives': 199,
   'total_samples': 1271

In [ ]:
results_per_outage[results_per_outage['market_month'] == '2025-11'].xs('316316', level='constraint_id')

,70_diff,90,105,curvature_100,80_diff,95,prob_exceed_100,prob_exceed_90,prob_overload,season_cos,forecast_horizon,100_diff,100,95_diff,85_diff,110,risk_ratio,log_density_100,60_diff,season_sin,90_diff,105_diff,actual_shadow_price,branch_name,outage_date,auction_month,market_month,predicted_shadow_price,binding_probability,binding_probability_scaled,threshold,predicted_binding,model_used,actual_binding
flow_direction,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1.784015,3.258502,5.754621,-0.148212,2.640703,3.908352,8.430245,7.388001,8.430245,1.250873,-0.371556,5.138524,4.505723,4.280096,3.077991,6.537937,-0.002386,4.963730,0.926014,-1.162754,3.615944,6.282474,3622.70,AUST_TAYS_1545 A,2025-11-01,2025-10-01,2025-11-01,910.410828,0.997876,0.997876,0.5,1,Anomaly: AUST_TAYS_1545 A,1
1,1.734512,3.213132,5.700590,-0.449658,2.599568,3.863028,8.784135,7.622319,8.784135,1.250873,-0.371556,5.156673,4.578977,4.288577,3.039562,6.515162,-0.002385,5.023914,0.882424,-1.162754,3.569837,6.241609,8037.55,AUST_TAYS_1545 A,2025-11-04,2025-10-01,2025-11-01,864.286865,1.000000,1.000000,0.5,1,Anomaly: AUST_TAYS_1545 A,1
1,1.862594,3.459646,5.726164,-0.247413,2.660966,4.022102,7.835947,7.050664,7.835947,1.250873,-0.371556,5.276386,4.767611,4.465211,3.143225,6.787118,-0.002388,5.176953,0.908692,-1.162754,3.779701,6.383535,1638.56,AUST_TAYS_1545 A,2025-11-07,2025-10-01,2025-11-01,992.354065,1.000000,1.000000,0.5,1,Anomaly: AUST_TAYS_1545 A,1
1,1.759646,3.039187,5.304565,-0.423014,2.416127,3.556261,7.938228,6.943010,7.938228,1.250873,-0.371556,4.892203,4.424071,4.044664,2.833966,6.452069,-0.002386,4.896139,0.793799,-1.162754,3.331057,5.990821,5366.23,AUST_TAYS_1545 A,2025-11-10,2025-10-01,2025-11-01,835.852417,0.993511,0.993511,0.5,1,Anomaly: AUST_TAYS_1545 A,1
1,1.499928,3.075816,5.564916,0.090726,2.370683,3.617446,9.700059,8.159123,9.700059,1.250873,-0.371556,4.933825,4.295742,4.019583,2.898123,6.569988,-0.002380,4.788806,0.706745,-1.162754,3.379748,6.191569,2126.53,AUST_TAYS_1545 A,2025-11-13,2025-10-01,2025-11-01,1077.029053,0.986651,0.986651,0.5,1,Anomaly: AUST_TAYS_1545 A,1
1,-0.471639,-0.320627,-0.216123,-0.131973,-0.393982,-0.276470,-0.182346,-0.242497,-0.182346,1.250873,-0.371556,-0.232826,-0.238561,-0.265645,-0.349425,-0.181014,-0.002412,-0.283252,-0.588884,-1.162754,-0.305396,-0.205661,0.00,AUST_TAYS_1545 A,2025-11-16,2025-10-01,2025-11-01,0.000000,0.000000,0.000000,0.5,0,Never-Binding: AUST_TAYS_1545 A,0
1,-0.471639,-0.320627,-0.216123,-0.131973,-0.393982,-0.276470,-0.182346,-0.242497,-0.182346,1.250873,-0.371556,-0.232826,-0.238561,-0.265645,-0.349425,-0.181014,-0.002412,-0.283252,-0.588884,-1.162754,-0.305396,-0.205661,0.00,AUST_TAYS_1545 A,2025-11-19,2025-10-01,2025-11-01,0.000000,0.000000,0.000000,0.5,0,Never-Binding: AUST_TAYS_1545 A,0
1,1.683353,3.157262,5.500393,-0.572884,2.254989,3.834945,6.262102,5.856624,6.262102,1.250873,-0.371556,4.926953,4.333540,4.159164,2.827407,5.738423,-0.002390,4.820561,1.009203,-1.162754,3.525989,5.765238,0.00,AUST_TAYS_1545 A,2025-11-22,2025-10-01,2025-11-01,869.166748,0.988672,0.988672,0.5,1,Anomaly: AUST_TAYS_1545 A,0
1,1.664688,3.139190,5.396115,-0.318122,2.289832,3.745347,6.177483,5.768200,6.177483,1.250873,-0.371556,4.817300,4.223107,4.057738,2.835403,5.732501,-0.002391,4.727447,1.019168,-1.162754,3.474266,5.704220,0.00,AUST_TAYS_1545 A,2025-11-25,2025-10-01,2025-11-01,833.698486,0.982768,0.982768,0.5,1,Anomaly: AUST_TAYS_1545 A,0


In [ ]:
final_results.xs('316316', level='constraint_id')

,branch_name,105,80_diff,prob_exceed_100,100_diff,curvature_100,log_density_100,100,105_diff,risk_ratio,season_cos,95,85_diff,season_sin,90_diff,60_diff,95_diff,forecast_horizon,70_diff,110,90,prob_overload,prob_exceed_90,actual_shadow_price,predicted_shadow_price,binding_probability,binding_probability_scaled,threshold,predicted_binding_count,actual_binding,auction_month,market_month,model_used,predicted_binding,error,abs_error
flow_direction,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,AUST_TAYS_1545 A,-0.222676,-0.396567,-0.189749,-0.240615,-0.132835,-0.291872,-0.247593,-0.213415,-0.003436,-1.638969,-0.287794,-0.359034,-0.302390,-0.314818,-0.572481,-0.274367,-0.978030,-0.473142,-0.190663,-0.328508,-0.189749,-0.250259,0.00,0.000000,0.000000,0.000000,0.5,0,0,2025-06-01,2025-06-01,Never-Binding: AUST_TAYS_1545 A,False,0.000000,0.000000
1,AUST_TAYS_1545 A,-0.222891,-0.388058,-0.189902,-0.240680,-0.129611,-0.291613,-0.247692,-0.213674,-0.003438,-1.441020,-0.286328,-0.347425,-1.022871,-0.312816,-0.541009,-0.273542,-0.977787,-0.457775,-0.191285,-0.326137,-0.189902,-0.249779,0.00,0.000000,0.000000,0.000000,0.5,0,0,2025-07-01,2025-07-01,Never-Binding: AUST_TAYS_1545 A,False,0.000000,0.000000
1,AUST_TAYS_1545 A,-0.222892,-0.399015,-0.189920,-0.240763,-0.134025,-0.291833,-0.247836,-0.213674,-0.003438,-0.902305,-0.287665,-0.355453,-1.550330,-0.314779,-0.555862,-0.274351,-0.586052,-0.466436,-0.191285,-0.328569,-0.189920,-0.250377,0.00,0.000000,0.000000,0.000000,0.5,0,0,2025-07-01,2025-08-01,Never-Binding: AUST_TAYS_1545 A,False,0.000000,0.000000
1,AUST_TAYS_1545 A,1.750061,1.255955,1.999085,1.717400,0.003636,1.929407,1.642664,1.824497,-0.003417,-0.166407,1.566034,1.435103,-1.743393,1.544866,0.531110,1.629355,-0.194316,0.904382,1.819975,1.483169,1.999085,2.010755,0.00,3093.115051,0.517068,0.517068,0.5,5,0,2025-07-01,2025-09-01,Never-Binding: AUST_TAYS_1545 A,True,3093.115051,3093.115051
1,AUST_TAYS_1545 A,3.587550,2.317973,4.510896,3.417564,-0.155593,3.727192,3.186764,3.760944,-0.003389,0.569491,2.868534,2.587164,-1.550330,2.812326,1.132692,3.066830,0.197420,1.814445,3.775946,2.685187,4.510896,4.275150,2950.54,10312.792114,0.932455,0.932455,0.5,11,1,2025-07-01,2025-10-01,Anomaly: AUST_TAYS_1545 A,True,7362.252114,7362.252114
1,AUST_TAYS_1545 A,3.988913,1.834263,5.583727,3.670722,-0.253874,3.659353,3.316810,4.425398,-0.003382,1.108206,2.765243,2.227568,-1.022871,2.602017,0.589885,3.069682,0.589156,1.272798,4.726376,2.391090,5.583727,4.962689,20791.57,0.000000,0.790102,0.790102,0.5,8,1,2025-07-01,2025-11-01,Anomaly: AUST_TAYS_1545 A,True,-20791.570000,20791.570000


In [ ]:
final_results.sort_values(['actual_shadow_price', 'binding_probability_scaled'], ascending=False).drop_duplicates(subset=['branch_name', 'auction_month']).head(20)

,,branch_name,log_density_100,95_diff,105_diff,100_diff,95,80_diff,90_diff,prob_overload,prob_exceed_90,prob_exceed_100,70_diff,85_diff,110,105,curvature_100,90,60_diff,100,risk_ratio,actual_shadow_price,predicted_shadow_price,binding_probability,binding_probability_scaled,threshold,predicted_binding_count,actual_binding,auction_month,market_month,model_used,predicted_binding,error,abs_error
constraint_id,flow_direction,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
322296,1,FORMAFORMN11_1 1,-0.298647,-0.277988,-0.218690,-0.246380,-0.286987,-0.355082,-0.311701,-0.193522,-0.252338,-0.193522,-0.398123,-0.339996,-0.188941,-0.231566,-0.111418,-0.322911,-0.304735,-0.248626,-0.002476,59113.56,19572.664551,0.755965,0.530701,0.74,11,1,2025-10-01,2025-10-01,Branch: FORMAFORMN11_1 1,True,-39540.895449,39540.895449
373761,1,GOOSEMNPIP11_1 1,-0.298647,-0.277990,-0.218690,-0.246380,-0.286990,-0.404167,-0.316386,-0.193522,-0.253349,-0.193522,-0.483452,-0.358930,-0.188941,-0.231566,-0.127217,-0.331435,-0.600108,-0.248626,-0.002476,49015.21,0.000000,0.040880,0.021516,0.95,0,1,2025-10-01,2025-10-01,Branch: GOOSEMNPIP11_1 1,False,-49015.210000,49015.210000
191486,1,NEBRCSUB3434_1 1,-0.168057,-0.193604,-0.187109,-0.183987,-0.208883,0.050854,-0.212375,-0.174093,-0.201605,-0.174093,0.284406,-0.120213,-0.160122,-0.199768,-0.092672,-0.209068,0.579188,-0.165461,-0.009854,45969.48,3591.270874,0.445176,0.339059,0.67,2,1,2025-08-01,2025-08-01,Branch: NEBRCSUB3434_1 1,True,-42378.209126,42378.209126
276166,1,MNTCELO TR6__2,5.864989,6.217540,5.819640,6.139771,6.060308,4.987471,6.063371,3.857620,5.132705,3.857620,3.730285,5.555863,5.115379,6.055116,-0.221620,5.875901,2.122431,5.961734,-0.002474,41374.66,3245.228394,0.170138,0.170138,0.50,2,1,2025-11-01,2025-11-01,Anomaly: MNTCELO TR6__2,True,-38129.431606,38129.431606
233584,1,BIGSTBROWN23_1 1,0.983481,0.869712,0.394810,0.615527,0.935809,1.732736,1.212618,0.238637,0.587311,0.238637,2.303848,1.553061,0.321171,0.434017,0.977879,1.410276,2.431939,0.728137,-0.002482,37863.92,8389.231873,0.744353,0.581395,0.70,8,1,2025-11-01,2025-11-01,Branch: BIGSTBROWN23_1 1,True,-29474.688127,29474.688127
337116,1,MRYVLBRADD16_1 1,0.351470,0.178229,0.181563,0.198473,0.161653,0.027477,0.123208,0.115154,0.138687,0.115154,0.014836,0.062438,0.146684,0.201199,-0.215639,0.087242,0.117517,0.189641,-0.009706,28938.79,3379.974396,0.613861,0.458962,0.70,6,1,2025-07-01,2025-07-01,Branch: MRYVLBRADD16_1 1,True,-25558.815604,25558.815604
121700,-1,ULRICMAHNO11_1 1,0.296960,0.250742,0.178339,0.217300,0.265963,0.209076,0.277268,0.085288,0.164429,0.085288,0.080594,0.251096,0.131488,0.208885,0.064029,0.278146,-0.136753,0.215745,-0.002455,26006.01,0.000000,0.226617,0.147154,0.77,0,1,2025-10-01,2025-10-01,Branch: ULRICMAHNO11_1 1,False,-26006.010000,26006.010000
232780,1,RILLA__RIVTON3 A,5.368736,5.474837,5.491588,5.374908,5.503908,4.517072,5.405215,5.150349,5.746467,5.150349,3.200098,5.086503,5.085989,5.511927,0.698298,5.150596,1.609185,5.074117,-0.002459,25956.94,6012.525391,0.477386,0.350037,0.69,2,1,2025-10-01,2025-10-01,Branch: RILLA__RIVTON3 A,True,-19944.414609,19944.414609
430652,1,ROSEHURON23_1 1,0.321483,0.271129,-0.045270,0.092041,0.340501,0.969207,0.493305,-0.067443,0.108055,-0.067443,1.155740,0.788643,-0.076464,-0.014791,0.611249,0.607867,1.321066,0.169871,-0.002474,25655.66,8252.693359,0.532465,0.445224,0.62,3,1,2025-10-01,2025-10-01,Branch: ROSEHURON23_1 1,True,-17402.966641,17402.966641
